# Chapter 1 Companion: The Five Principles of Prompting

*A hands-on companion to Chapter 1 of* Prompt Engineering for Generative AI. *Read the chapter first, then work through this notebook.*

**What you will learn**

By the end of this notebook you will be able to:

1. **Give Direction**: tell the model who it is writing for and what good looks like.
2. **Specify Format**: get answers back in a shape your code can use.
3. **Provide Examples**: steer the model by showing, not just telling.
4. **Evaluate Quality**: measure whether one prompt is really better than another.
5. **Divide Labor**: split a big task into smaller, more reliable steps.

**How to use this notebook**

- Run the cells **in order**, top to bottom. Click **▷** to the left of a cell to run it.
- Cells marked **🧪 Try it** call GPT. Each one says how many requests it sends.
- Cells marked **✍️ Your turn** ask you questions. Click **▷**, then read each question printed **below the cell**, type your answer in the **box at the top of VS Code** and press **Enter**. No code editing needed.
- The whole notebook sends about **15 requests** to `gpt-6-luna`. Expected cost: a fraction of one cent.
- A built-in safety cap stops the notebook after **25 requests**, so a mistake can't run up charges.

## Setup

This cell connects to OpenAI and creates two helpers you'll use throughout:

- `ask(prompt)` sends one prompt to GPT and returns the answer as text.
- `show(label, text)` prints an answer with a heading, so outputs are easy to compare.

Your API key is read from the `OPENAI_API_KEY` Windows setting. It is never shown.

In [ ]:
import os
import json
from openai import OpenAI

# ---- Settings ----
MODEL = "gpt-6-luna"     # OpenAI's low-cost model (checked Nov 2026)
MAX_REQUESTS = 25        # safety cap for this whole notebook
# -------------------

if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is not set. Nothing was sent to OpenAI.")

client = OpenAI()        # reads the key from the environment
requests_made = 0

def ask(prompt, system="You are a helpful assistant."):
    # Send one prompt to GPT and return its answer as text.
    # Every call is one paid request (counted against MAX_REQUESTS).
    global requests_made
    if requests_made >= MAX_REQUESTS:
        raise RuntimeError(f"Safety cap reached ({MAX_REQUESTS} requests). "
                           "Raise MAX_REQUESTS in the Setup cell if you want more.")
    requests_made += 1
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": prompt},
        ],
        # Skip the model's hidden "thinking" step: cheaper and faster.
        extra_body={"reasoning_effort": "none"},
    )
    return response.choices[0].message.content

def question(number, total, text, example):
    # Ask you one question. The question is printed clearly below the cell,
    # and you type your answer in the box that opens at the TOP of VS Code.
    print(f"QUESTION {number} of {total}: {text}")
    print(f"   Example: {example}")
    print("   -> Type your answer in the box at the TOP of the window, then press Enter.")
    print()
    return input(f"Answer {number} of {total}: {text}")

def show(label, text):
    print(f"--- {label} ---")
    print(text)
    print()

print(f"Ready. Model: {MODEL}. Requests used so far: {requests_made}/{MAX_REQUESTS}")

## 1. Give Direction

A model knows a little about almost everything, but nothing about *your* situation. If you ask for "a product name", it has to guess the audience, the tone, the length and the purpose. It usually guesses something safe and generic.

**Giving direction** means filling in those blanks yourself:

| Blank the model would otherwise guess | Example of direction |
|---|---|
| Who is it for? | "for busy parents" |
| What tone or style? | "playful, like a children's brand" |
| What constraints? | "one or two words, easy to spell" |
| What does success look like? | "memorable enough to say aloud once and recall later" |

A useful mental model: brief the AI the way you'd brief a capable new employee on their first day. They're smart, but they don't know your business yet.

### 🧪 Try it: a vague prompt vs. a directed prompt (2 requests)

Same task, two prompts. Compare how specific and usable the answers are.

In [ ]:
vague_prompt = "Suggest product names for shoes."

directed_prompt = (
    "Suggest product names for a shoe that adjusts to fit any foot size. "
    "Audience: parents buying shoes for fast-growing kids. "
    "Tone: friendly and trustworthy, not technical. "
    "Constraints: one or two words, easy to spell and say aloud."
)

show("Vague prompt", ask(vague_prompt))
show("Directed prompt", ask(directed_prompt))

**What to notice:** the vague answer is usually long and generic, aimed at nobody in particular. The directed answer should fit the audience and obey the constraints. Same model, same cost; the only difference is the information you gave it.

### ✍️ Your turn (1 request)

Pick a product or service you know well, perhaps from your own work. Click **▷**, then answer the four questions in the box at the **top of VS Code**, pressing **Enter** after each. The cell builds a directed prompt from your answers and sends it.

*If you leave an answer blank, nothing is sent; just run the cell again.*

In [ ]:
# Each question appears below this cell; type answers in the box at the TOP of VS Code.
product     = question(1, 4, "What are you naming?", "a one-page AI cheat sheet")
audience    = question(2, 4, "Who is it for?", "warehouse supervisors new to AI")
tone        = question(3, 4, "What tone should the names have?", "practical, no hype")
constraints = question(4, 4, "Any constraints?", "2-4 words, no jargon")

if not all(answer.strip() for answer in [product, audience, tone, constraints]):
    print("One or more answers were blank, so nothing was sent. Run the cell again.")
else:
    # Your answers are dropped into a directed prompt (audience + tone + constraints)
    my_directed_prompt = (
        f"Suggest names for {product}. "
        f"Audience: {audience}. "
        f"Tone: {tone}. "
        f"Constraints: {constraints}."
    )
    print("Your prompt:\n" + my_directed_prompt + "\n")
    show("My directed prompt", ask(my_directed_prompt))

## 2. Specify Format

People can read an answer in any shape. **Code can't.** As soon as a program uses the model's output (to save it, sort it or put it on a web page) the format becomes as important as the content.

Ways to specify format:

- Name the format outright: *a numbered list*, *a table*, *JSON*, *CSV*.
- Show the exact shape you want, with placeholder values.
- Say what to leave out: *"no introduction, no explanation, only the JSON."*

**JSON** (JavaScript Object Notation) is the most common choice when code needs the answer. It's plain text with a strict structure that Python can turn straight into a list or dictionary.

### 🧪 Try it: ask for JSON, then prove the code can use it (1 request)

`json.loads(...)` converts the text into a real Python list. If the model adds chatty extra text, that step fails, which is exactly why format instructions matter.

In [ ]:
format_prompt = (
    "Suggest 5 product names for a shoe that adjusts to fit any foot size. "
    'Respond with ONLY a JSON object in exactly this shape: {"names": ["name1", "name2"]}. '
    "No introduction, no explanation, no code fences."
)

raw_answer = ask(format_prompt)
show("Raw text from GPT", raw_answer)

try:
    names = json.loads(raw_answer)["names"]
    print("Parsed successfully. Python now has a real list:")
    for i, name in enumerate(names, start=1):
        print(f"  {i}. {name}")
except (json.JSONDecodeError, KeyError, TypeError) as error:
    print("Could not parse the answer as the JSON we asked for:", error)
    print("This is what happens when the format isn't followed.")

**Why this matters:** once the answer is a Python list, you can count it, sort it, save it or feed it into the next step automatically. That's the step from *chatting with AI* to *building with AI*.

## 3. Provide Examples

Instructions *tell* the model what you want; examples *show* it. Often showing is more powerful, because examples carry details that are hard to put into words: length, rhythm, style, level of creativity.

Vocabulary you'll see throughout the book:

| Term | Meaning |
|---|---|
| **Zero-shot** | No examples, just the instruction. |
| **One-shot** | One example. |
| **Few-shot** | Several examples (typically 2–5). |

**The trade-off:** examples make outputs more consistent, but they can also make them *less creative*, because the model tends to copy the examples closely. You saw this in the A/B test: the few-shot variant returned short, comma-separated names in the exact style of its examples.

### 🧪 Try it: zero-shot vs. few-shot (2 requests)

In [ ]:
task = "Product: a water bottle that reminds you to drink.\nNames:"

zero_shot = task

few_shot = (
    "Product: a lamp that dims itself at bedtime.\nNames: DuskGlow, SleepLight, NightEase\n\n"
    "Product: a backpack with a built-in phone charger.\nNames: ChargePack, PowerTrek, JuiceBag\n\n"
    + task
)

show("Zero-shot (no examples)", ask(zero_shot))
show("Few-shot (2 examples)", ask(few_shot))

**What to notice:** the few-shot answer almost always copies the pattern: three names, one line, CamelCase compound words. The zero-shot answer chooses its own format and length.

> **Rule of thumb:** use examples when consistency matters more than variety. Vary your examples (different lengths, styles) when you want the model to generalize instead of copy.

## 4. Evaluate Quality

How do you *know* a prompt is better? One good answer could be luck: the same prompt can produce a different answer every time you run it. Real evaluation means:

1. Running each prompt **several times**.
2. Scoring every answer against **clear criteria**.
3. Comparing the **average** scores.

Scores can come from people (the thumbs-up/down rating in the A/B notebook), from code (automatic checks), or from another model acting as a judge. Here we'll use **code**: it's objective, free to run, and needs no buttons.

Our criteria for a good answer:

- ✅ It is valid JSON that Python can read.
- ✅ It contains exactly 5 names.
- ✅ Every name is at most 2 words.

### 🧪 Try it: score two prompts automatically (6 requests)

Prompt A gives loose instructions; Prompt B specifies the format precisely. Each runs 3 times.

In [ ]:
import pandas as pd

RUNS_PER_PROMPT = 3   # each run = 1 request. 2 prompts x 3 runs = 6 requests

prompt_a = "Give me 5 names for a shoe that fits any foot size, as JSON."
prompt_b = (
    "Suggest exactly 5 names (each 1-2 words) for a shoe that fits any foot size. "
    'Respond with ONLY this JSON shape: {"names": ["...", "...", "...", "...", "..."]}'
)

def score(answer):
    # Return a score from 0 to 3, one point per criterion met.
    try:
        names = json.loads(answer)["names"]
    except (json.JSONDecodeError, KeyError, TypeError):
        return 0                                    # not valid JSON: no points
    points = 1                                      # valid JSON
    if len(names) == 5:
        points += 1                                 # exactly 5 names
    if all(len(str(n).split()) <= 2 for n in names):
        points += 1                                 # every name 1-2 words
    return points

results = []
for label, prompt in [("A (loose)", prompt_a), ("B (precise)", prompt_b)]:
    for run in range(1, RUNS_PER_PROMPT + 1):
        results.append({"prompt": label, "run": run, "score (0-3)": score(ask(prompt))})

df = pd.DataFrame(results)
print(df.to_string(index=False))
print()
print("Average score per prompt:")
print(df.groupby("prompt")["score (0-3)"].mean())
print(f"\nRequests used so far: {requests_made}/{MAX_REQUESTS}")

**What to notice:** the precise prompt should score a perfect or near-perfect 3. The loose prompt often loses points, for example by wrapping the JSON in extra text or code fences, which breaks parsing.

You've just done what the book calls *evaluating quality*: you turned "this one feels better" into a number you can compare.

## 5. Divide Labor

Ask a person to *research, write, edit and format a report all at once*, and quality drops. Models are the same. **Dividing labor** means breaking one big request into a chain of smaller steps, where each step's output becomes the next step's input.

Benefits:

- Each step is simpler, so it's more reliable.
- You can check, fix or swap out any single step.
- You can use different instructions (or even different models) for different steps.

The book returns to this idea in Chapter 4 (*prompt chaining* with LangChain) and Chapter 6 (*agents*).

### 🧪 Try it: a three-step chain (3 requests)

Step 1 brainstorms → Step 2 picks the best → Step 3 writes a tagline for the winner.

In [ ]:
product = "a shoe that adjusts to fit any foot size, for parents of fast-growing kids"

# Step 1: brainstorm
ideas = ask(f"List 6 short product names for {product}. One per line, no numbering.")
show("Step 1: Brainstorm", ideas)

# Step 2: judge (uses Step 1's output as its input)
choice = ask(
    "From the names below, choose the single best one for busy parents. "
    "Reply in exactly two lines: the name on line 1, a one-sentence reason on line 2.\n\n" + ideas
)
show("Step 2: Pick the best", choice)

# Step 3: create (uses Step 2's output)
best_name = choice.strip().splitlines()[0]
tagline = ask(f"Write one catchy tagline (under 10 words) for a product called '{best_name}', {product}.")
show("Step 3: Tagline", tagline)

**What to notice:** each step had one job. If you dislike the tagline, you rerun only Step 3, not the whole chain.

## Putting It All Together

Real prompts combine the principles. Here's one prompt using **direction**, **format** and **examples** at once. (Evaluation and division of labor happen *around* prompts like this one, as you saw above.)

### ✍️ Your turn: build a complete prompt (1 request)

Click **▷** and answer the questions in the box at the **top of VS Code**. The cell assembles a prompt that uses **direction** (your audience and tone), **examples** (names in a style you like) and **format** (JSON), then scores the answer with the same 0–3 checks as before.

In [ ]:
# Each question appears below this cell; type answers in the box at the TOP of VS Code.
product  = question(1, 4, "What are you naming?", "a one-page AI cheat sheet")
audience = question(2, 4, "Who is it for?", "warehouse supervisors new to AI")
tone     = question(3, 4, "What tone should the names have?", "practical, no hype")
examples = question(4, 4, "Two or three names in a style you like (comma-separated)", "QuickRef, ShiftSmart, FloorGuide")

if not all(answer.strip() for answer in [product, audience, tone, examples]):
    print("One or more answers were blank, so nothing was sent. Run the cell again.")
else:
    complete_prompt = (
        # 1. Direction: audience, tone, constraints
        f"You are naming {product} for {audience}. "
        f"Tone: {tone}. Each name is 1-2 words.\n\n"
        # 3. Examples: show the style you want
        f"Examples of the style we like: {examples}\n\n"
        # 2. Format: exactly what shape to return
        'Suggest 5 new names. Respond with ONLY JSON: {"names": ["..."]}'
    )
    print("Your prompt:\n" + complete_prompt + "\n")
    answer = ask(complete_prompt)
    show("Complete prompt", answer)
    if "score" in globals():
        print("Score (0-3):", score(answer))
    else:
        print("(Run the 4. Evaluate Quality cell first to see a score.)")
    print(f"Requests used: {requests_made}/{MAX_REQUESTS}")

## Summary

| Principle | One-line idea | What you did |
|---|---|---|
| **1. Give Direction** | Fill in what the model would otherwise guess. | Compared a vague prompt with a directed one. |
| **2. Specify Format** | Ask for a shape your code can use. | Got JSON and turned it into a Python list. |
| **3. Provide Examples** | Show, don't just tell. | Compared zero-shot with few-shot. |
| **4. Evaluate Quality** | Measure, don't guess. | Scored two prompts automatically over several runs. |
| **5. Divide Labor** | Split big tasks into a chain of small ones. | Built a three-step brainstorm → pick → tagline chain. |

These five principles run through every remaining chapter. When a prompt disappoints you, check them in order: *Did I give direction? Specify format? Show examples? Measure the result? Ask for too much at once?*

## Check Your Understanding

Answer in your own words, in your notes file, before opening the answers.

1. You ask for "a summary of this report" and get something too long and too technical. Which principle would you apply first, and what would you add?
2. Why does `json.loads` sometimes fail on a model's answer, even when the answer *looks* like JSON?
3. Your few-shot prompt produces names that copy your examples too closely. What can you change?
4. Prompt B scored 3/3 once. Why isn't that enough to call it better than Prompt A?
5. Name one advantage of splitting a task into a chain of steps.

<details>
<summary><b>Show answers</b></summary>

1. **Give Direction**: state the audience (e.g. "for a non-technical manager") and constraints (e.g. "under 100 words, plain English").
2. The model may add extra text, explanations or code fences around the JSON, and any extra characters make strict parsing fail. That's why the prompt says *ONLY JSON*.
3. Use more varied examples, use fewer examples, or add an instruction such as "be original; don't reuse words from the examples."
4. One run can be luck. Run each prompt several times and compare the **average** scores.
5. Any one of: each step is simpler and more reliable; you can check or rerun a single step; each step can have its own instructions or model.

</details>

## Next Steps

- Add today's key ideas to `notes/chapter-01.md`.
- **Chapter 2** explains *why* these principles work: how language models turn text into numbers and predict the next word.